# FRC-GAN fine-tuning (Real-ESRGAN x1 + FRC loss)

1. Setup  2. Training data  3. Fine-tuning (`frcgan_train.train_frcgan`)  4. Visual comparison  5. FRC loss check / evaluation

The FRC loss and the FRC-AUC validation metric are in `basicsr/basicsr/losses/frc_loss.py`, and the training wrapper is in `frcgan_train.py` (repo root).

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import glob
from pathlib import Path

import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt

print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 1. Paths
Change `DATA_ROOT` here. Every other path below is relative to it.

In [ ]:
DATA_ROOT = Path('/content/drive/MyDrive')          # e.g. Path(r'C:\Users\LocalAdmin\Box\...')
REALESRGAN_DATA = DATA_ROOT / 'realesrgan_data'
BRAIN_DATA = DATA_ROOT / 'Brain_data'

## 2. Training data (meta info for paired data)

In [ ]:
pair_root = REALESRGAN_DATA / 'x1_denoise' / 'Liver_all_addednoise' / 'selected'
meta_file = pair_root / 'meta_info' / 'meta_info_liver_isotope_selected.txt'
meta_file.parent.mkdir(parents=True, exist_ok=True)

!"{sys.executable}" scripts/generate_meta_info_pairdata.py --input "{pair_root / 'HR_selected'}" "{pair_root / 'LR_selected'}" --meta_info "{meta_file}"

## 3. Fine-tuning

`train_frcgan` starts from the base yml, fills in the FRC-GAN settings, saves the resolved options to `options/generated/<name>.yml` (with a header recording the time, git commit and arguments), and runs Real-ESRGAN training. Any parameter left as `None` keeps the base yml value. For any other setting, use `overrides={'a:b:c': value}`.

In [ ]:
from frcgan_train import train_frcgan

common = dict(
    dataroot_gt=pair_root, dataroot_lq=pair_root,      # meta info paths are relative to pair_root
    meta_info=meta_file,
    # pretrain_g='experiments/pretrained_models/RealESRGAN_x1plus.pth',   # None = keep base yml
    gt_usm=False,          # no unsharp-masking of the GT for quantitative data
    total_iter=50_000, lr=1e-4, batch_size=12, gt_size=256,
    checkpoint_freq=5_000,
    # val_gt=..., val_lq=..., val_freq=5_000,           # logs PSNR + FRC-AUC during training
)

# inspect the generated options without training
print(train_frcgan('FRCGAN_liver_frc0p1', frc_weight=0.1, dry_run=True, **common)['options'])

In [ ]:
run = train_frcgan('FRCGAN_liver_frc0p1', frc_weight=0.1, auto_resume=True, **common)
run

Ablation over the FRC weight (0 = L1 baseline). Each run gets its own option file and experiment folder.

In [ ]:
runs = {w: train_frcgan(f'FRCGAN_liver_frc{str(w).replace(".", "p")}', frc_weight=w, auto_resume=True, **common)
        for w in (0, 0.05, 0.1, 0.2)}

## 4. Visual comparison

In [ ]:
def imread(path, gray=True):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE if gray else cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    return img if gray else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def list_images(folder):
    return sorted(glob.glob(os.path.join(str(folder), '*')))


def show_comparison(folders, gray=True, indices=range(4), cmap='magma', size=10):
    """folders: {title: folder}. gray=False shows RGB. Draws one row per index, one column per folder."""
    lists = {k: list_images(v) for k, v in folders.items()}
    n_min = min(len(v) for v in lists.values())
    if n_min == 0:
        raise ValueError({k: len(v) for k, v in lists.items()})
    indices = [i for i in indices if i < n_min]
    for i in indices:
        fig, axes = plt.subplots(1, len(folders), figsize=(size * len(folders), size))
        for ax, (title, files) in zip(np.atleast_1d(axes), lists.items()):
            ax.imshow(imread(files[i], gray), cmap=cmap if gray else None)
            ax.set_title(f'{title}  [{os.path.basename(files[i])}]', fontsize=14)
            ax.axis('off')
        plt.tight_layout()
        plt.show()

In [ ]:
COMPARISONS = {
    'checkpoints (liver)': ({'gt': REALESRGAN_DATA / 'HR_train',
                             '40w5': REALESRGAN_DATA / 'RES_405000',
                             '33w5': REALESRGAN_DATA / 'RES_335000',
                             '18w': REALESRGAN_DATA / 'RES_180000'}, True),
    'brain: methods': ({'Input': BRAIN_DATA / 'LR_high_SNR',
                        'Real-ESRGAN res': BRAIN_DATA / 'HR_realesrgan',
                        'Ground truth': BRAIN_DATA / 'HR_high_SNR',
                        'ESRGAN': BRAIN_DATA / 'RES_high_SNR',
                        'Real-ESRGAN enh': BRAIN_DATA / 'ENH_realesrgan'}, False),
    'brain: L1 vs FRC': ({'L1 res': BRAIN_DATA / 'HR_realesrgan',
                          'L1 enh': BRAIN_DATA / 'ENH_high_SNR',
                          'FRC res': BRAIN_DATA / 'HR_frc',
                          'FRC enh': BRAIN_DATA / 'ENH_frc'}, True),
    'brain: test set': ({'L1 test': BRAIN_DATA / 'RES_2_realesrgan',
                         'FRC test': BRAIN_DATA / 'RES_2_frc',
                         'Ground truth': BRAIN_DATA / 'HR_2'}, True),
}

In [ ]:
show_comparison(*COMPARISONS['checkpoints (liver)'], indices=[0, 1, 2, 3, 3300, 3301, 3302])

In [ ]:
show_comparison(*COMPARISONS['brain: methods'], indices=[0, 1, 2, 3, 10, 11, 12, 13])

In [ ]:
show_comparison(*COMPARISONS['brain: L1 vs FRC'], indices=[0, 1, 2, 3, 10, 11, 12, 100, 101, 102])

In [ ]:
show_comparison(*COMPARISONS['brain: test set'], indices=[0, 1, 2, 3])

## 5. FRC loss check and evaluation
Quick sanity checks for the loss: identical images give 0, more noise gives a higher loss, and gradients stay finite. After that, the FRC-AUC of each method's outputs against the ground truth (higher is better).

In [ ]:
from basicsr.losses.frc_loss import FRCLoss, frc_curve, frc_loss

gt = torch.rand(2, 1, 256, 256)
print('identical  :', frc_loss(gt, gt).item())
for s in (0.1, 0.3, 0.5):
    print(f'noise {s:.1f}  :', round(frc_loss(gt + s * torch.randn_like(gt), gt).item(), 4))
p = (gt + 0.3 * torch.randn_like(gt)).requires_grad_()
FRCLoss()(p, gt).backward()
print('grad finite:', torch.isfinite(p.grad).all().item())

In [ ]:
def frc_auc_folder(pred_folder, gt_folder, max_images=200, window=True, device='cpu'):
    """Mean FRC-AUC (0..1) of prediction images against ground truth, matched by sorted order."""
    preds, gts = list_images(pred_folder)[:max_images], list_images(gt_folder)[:max_images]
    aucs, curves = [], []
    for fp, fg in zip(preds, gts):
        a = torch.from_numpy(imread(fp).astype(np.float32) / 255.).to(device)
        b = torch.from_numpy(imread(fg).astype(np.float32) / 255.).to(device)
        frc, freq = frc_curve(a, b, window=window)
        curves.append(frc[0].cpu())
        aucs.append(torch.trapezoid(frc[0], freq).item())
    return np.mean(aucs), np.std(aucs), freq.cpu(), torch.stack(curves).mean(0)


gt_folder = BRAIN_DATA / 'HR_2'
plt.figure(figsize=(6, 4))
for name, folder in {'L1': BRAIN_DATA / 'RES_2_realesrgan', 'FRC': BRAIN_DATA / 'RES_2_frc'}.items():
    m, s, freq, curve = frc_auc_folder(folder, gt_folder)
    print(f'{name}: FRC-AUC = {m:.4f} +/- {s:.4f}')
    plt.plot(freq, curve, label=f'{name} (AUC {m:.3f})')
plt.xlabel('normalised spatial frequency'); plt.ylabel('FRC'); plt.ylim(-0.1, 1.05)
plt.legend(); plt.grid(alpha=0.3); plt.show()